# AGILAB API first run in Colab (Source Checkout)

This notebook shows the smallest source-checkout AGILAB API path from Google Colab.
It clones the repository, prepares an isolated Colab runtime venv under `/content`, and runs the built-in
`mycode_project` locally through `AgiEnv` and `AGI.run(...)` without mutating the base Colab kernel packages.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThalesGroup/agilab/blob/main/src/agilab/examples/notebook_quickstart/agi_core_colab_first_run_source.ipynb)


In [ ]:
# Source-checkout Colab path: clone AGILAB and prepare an isolated runtime venv.
import json
import shutil
import socket
import subprocess
import sys
from pathlib import Path

REPO_ROOT = Path("/content/agilab")
VENV_ROOT = Path("/content/.agilab_colab_source_env")
STATE_FILE = Path("/content/.agilab_colab_source_state.json")
STATE_VERSION = 7


def require_colab_internet(*hosts):
    missing = []
    for host in hosts:
        try:
            socket.getaddrinfo(host, 443)
        except OSError:
            missing.append(host)
    if missing:
        hosts_text = ", ".join(missing)
        raise SystemExit(
            f"Colab Internet appears to be unavailable for this notebook session; cannot reach {hosts_text}. Reconnect the runtime network, then rerun Cell 1."
        )


def ensure_uv():
    result = subprocess.run(
        [sys.executable, "-m", "pip", "show", "uv"],
        check=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    if result.returncode != 0:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)


def venv_python(venv_root: Path) -> Path:
    return venv_root / "bin" / "python"


def purelib_for(venv_python_path: Path) -> str:
    return subprocess.run(
        [
            str(venv_python_path),
            "-c",
            "import sysconfig; print(sysconfig.get_paths()['purelib'])",
        ],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


require_colab_internet("github.com", "pypi.org")

if (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_ROOT, check=True)
else:
    shutil.rmtree(REPO_ROOT, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ThalesGroup/agilab.git", str(REPO_ROOT)], check=True)

head = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT, check=True, capture_output=True, text=True).stdout.strip()
state = {}
if STATE_FILE.is_file():
    try:
        state = json.loads(STATE_FILE.read_text())
    except Exception:
        state = {}

venv_python_path = venv_python(VENV_ROOT)
needs_install = (
    state.get("version") != STATE_VERSION
    or state.get("mode") != "source"
    or state.get("head") != head
    or not venv_python_path.is_file()
)

if needs_install:
    ensure_uv()
    shutil.rmtree(VENV_ROOT, ignore_errors=True)
    subprocess.run([
        "uv", "venv", "--system-site-packages", str(VENV_ROOT), "--python", sys.executable,
    ], check=True)
    venv_python_path = venv_python(VENV_ROOT)
    subprocess.run([
        "uv", "pip", "install", "--python", str(venv_python_path), "-q",
        "-e", "/content/agilab/src/agilab/core/agi-env",
        "-e", "/content/agilab/src/agilab/core/agi-node",
        "-e", "/content/agilab/src/agilab/core/agi-cluster",
        "-e", "/content/agilab/src/agilab/core/agi-core",
    ], check=True)
    purelib = purelib_for(venv_python_path)
    STATE_FILE.write_text(json.dumps({
        "version": STATE_VERSION,
        "mode": "source",
        "head": head,
        "venv_root": str(VENV_ROOT),
        "venv_python": str(venv_python_path),
        "purelib": purelib,
    }))
    print("AGILAB source runtime prepared in an isolated Colab venv.")
else:
    print("AGILAB source runtime already prepared in an isolated Colab venv.")



In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

STATE_FILE = Path("/content/.agilab_colab_source_state.json")
if not STATE_FILE.is_file():
    raise RuntimeError("Run Cell 1 first to prepare the isolated Colab source runtime.")
try:
    _colab_state = json.loads(STATE_FILE.read_text())
except Exception as exc:
    raise RuntimeError("Colab source runtime state is unreadable. Rerun Cell 1.") from exc

REPO_ROOT = Path("/content/agilab")
PURELIB = _colab_state.get("purelib")
if not PURELIB:
    raise RuntimeError("Colab source runtime metadata is missing `purelib`. Rerun Cell 1.")
if PURELIB not in sys.path:
    sys.path.insert(0, PURELIB)

os.environ["IS_SOURCE_ENV"] = "1"
os.environ["AGI_CLUSTER_ENABLED"] = "0"
os.environ.pop("IS_WORKER_ENV", None)
os.environ["PYTHONNOUSERSITE"] = "1"
os.environ.pop("PYTHONSTARTUP", None)

for name in list(sys.modules):
    if name == "agilab" or name.startswith(("agilab.", "agi_env", "agi_env.", "agi_node", "agi_node.", "agi_cluster", "agi_cluster.")):
        del sys.modules[name]
importlib.invalidate_caches()

for entry in reversed([
    REPO_ROOT / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-env" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-node" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-cluster" / "src",
    REPO_ROOT / "src" / "agilab" / "core" / "agi-core" / "src",
]):
    entry_str = str(entry)
    if entry_str not in sys.path:
        sys.path.insert(0, entry_str)

from agi_cluster.agi_distributor import AGI, RunRequest
from agi_env import AgiEnv

APPS_PATH = REPO_ROOT / "src" / "agilab" / "apps"
BUILTIN_ROOT = APPS_PATH / "builtin"
APP = "mycode_project"
LOG_ROOT = Path.home() / "log" / "execute" / "mycode"

def worker_env_ready(app_env):
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if not worker_venv.exists():
        return False
    cmd = ["uv", "--quiet", "run", "--no-sync", "--project", str(worker_venv.parent)]
    pyvers_worker = getattr(app_env, "pyvers_worker", None)
    if pyvers_worker:
        cmd.extend(["--python", str(pyvers_worker)])
    cmd.extend(["python", "-c", "import agi_env, agi_node"])
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=False)
    return result.returncode == 0

async def install_if_needed(app_env, *, scheduler="127.0.0.1", workers=None, modes_enabled=0):
    if workers is None:
        workers = {"127.0.0.1": 1}
    worker_venv = Path.home() / "wenv" / f"{app_env.target}_worker" / ".venv"
    if worker_env_ready(app_env):
        return False
    action = "Installing"
    if worker_venv.parent.exists():
        shutil.rmtree(worker_venv.parent, ignore_errors=True)
        action = "Reinstalling"
    print(f"{action} worker for {app_env.app}...")
    await AGI.install(app_env, scheduler=scheduler, workers=workers, modes_enabled=modes_enabled)
    return True

print("Repo root:", REPO_ROOT)
print("Apps path:", APPS_PATH)
print("Builtin root:", BUILTIN_ROOT)
print("App:", APP)
print("Log root:", LOG_ROOT)



In [ ]:
app_env = AgiEnv(apps_path=APPS_PATH, app=APP, verbose=0)
await install_if_needed(app_env, scheduler="127.0.0.1", workers={"127.0.0.1": 1}, modes_enabled=0)
result = await AGI.run(
    app_env,
    request=RunRequest(scheduler="127.0.0.1", workers={"127.0.0.1": 1}, mode=0),
)
result
